## **Echantillonage probabiliste avec Python: Implémentation du sondage aléotoire simple, de la stratification et du sondage à probabilités inégales**


### **Fonctions necessaires à l'implémentation**

In [1111]:
# Fonctions utilitaires
def max_key_dic(dic):
    if isinstance(dic, dict):
        max_v=max(dic.values())
        for key in dic.keys():
            if float(dic[key])>=max_v:
                max_v=dic[key]
                cle=key
        return cle,max_v
    else:
        print("L'entrée n'est pas un dictionnaire")
        return None,None
dic={
    "joel":31,
    "thomas":100,
    "jean":27,
    "pitou":24,
    "Sena":2
}
print(max_key_dic(dic))

('thomas', 100)


In [1087]:
def arrondi_hamilton(dic):
    if isinstance(dic,dict):
        taille=int(sum(dic.values()))
        dic_int={}
        dic_reste={}
        for key in dic.keys():
            dic_int[key]=math.floor(dic[key])
            dic_reste[key]=dic[key]-dic_int[key]
        if sum(dic_int.values())==taille:
            return dic_int
        elif sum(dic_int.values())<taille:
            n_reste=taille-sum(dic_int.values())
            while n_reste>0:
                key_max,max_dic=max_key_dic(dic_reste)
                dic_int[key_max]=dic_int[key_max]+1
                del dic_reste[key_max]
                n_reste=n_reste-1
            return dic_int
        else:
            n_reste=sum(dic_int.values())-taille
            while n_reste>0:
                key_max,max_dic=max_key_dic(dic_reste)
                dic_int[key_max]=dic_int[key_max]-1
                del dic_reste[key_max]
                n_reste=n_reste-1
            return dic_int

In [1097]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import warnings

warnings.filterwarnings('ignore')


def equi_inclusion(df,u):
    """
    Cette fonction permet de recuper l'indexe du 
    plus petit élement d'une serie iméditement supérieur à une valuer donnée.
    """
    db=df-u
    index=db[db==db[db>=0].min()].index
    return index[0]


def select_sys(base_sondage,
               taille,
               poids=None,
               random_state=42
              ):
    """
    fonction pour selectionnner un échantillon probabiliste suivant la 
    méthode de tirage systématique.
    Il est demandé de préciser outre la base de sondage (DataFrame) et 
    les colonnes identifiant et nom des unités, le pas qui est le random state 
    et la taille de l'échantillon. 
    """
    import pandas as pd
    import numpy as np
    import math

    # génération du nombre aléatoire
    nb_alea=np.random.default_rng(seed=random_state).random()


    # vérifier si le paramètre poids est None. S'il est None c'est un sondage à probabilité égale
    
    # 1er cas: probabilité égale:
    
    # création des numéros d'ordres
    base_sondage["id_sys"]=range(1,len(base_sondage)+1)
    
    if poids==None:
        if taille>= len(base_sondage):
            print("la taille de l'échantillon est supérieur ou égal à la taille de la base de sondage")
        else:
            k=len(base_sondage)/taille if taille!=0 else 0
            select=[]
            for i in range(taille):
                select.append(math.floor(k*nb_alea+(i*k))+1)
            df = base_sondage[base_sondage["id_sys"].isin(select)]
            return select,df

    # 2er cas: probabilité inégale:
    else:
        c_poids=base_sondage[poids].apply(pd.to_numeric,errors="coerce").sum()
        for idx,row in base_sondage.iterrows():
            base_sondage.loc[idx,"poids_relatif"]=base_sondage.loc[idx,poids].sum()/c_poids
            base_sondage.loc[idx,"p_inclusion"]=base_sondage.loc[idx,"poids_relatif"]*taille
            base_sondage.loc[idx,"cum_p_inclusion"]=base_sondage.loc[0:idx,"p_inclusion"].sum()
        
        p_inclusion=base_sondage["p_inclusion"]>1
        # vérifier que les probabilié d'inclusion sont toutes inférieur à <1
        if p_inclusion.sum()==0:
            select=[nb_alea+i for i in range(taille)]
            select_cor=[equi_inclusion(base_sondage["cum_p_inclusion"],u) for u in select]
            df = base_sondage[base_sondage.index.isin(select_cor)]
            return select_cor,df
        else:
            select_cor_p=[next(u for u in p_inclusion if u==False),None]
            if len(select_cor_p)==taille:
                df=base_sondage[base_sondage["cum_p_inclusion"].isin(select_cor_p)]
                return select_cor_p,df
            else:
                long_reste=taille-len(select_cor_p)
                select=[nb_alea+i for i in range(long_reste)]
                select_cor=[equi_inclusion(base_sondage["cum_p_inclusion"],u) for u in select]
                df = base_sondage[base_sondage.index.isin(select_cor+select_cor_p)]
                select=select_cor+select_cor_p
                return select, df
## prise en compte de la stratification dans l'échantillonage

def select_sys_complete(base_sondage, 
                        taille, 
                        strate,
                        poids_u=None, # précise le type de sondage. si poids_u précisé alors sondage à probabilité inégale
                        poids=None,
                        random_state=42, 
                        share=0
                       ):
    """
    cette fonction permet de prendre en compte la stratification dans le plan de sondage
    les paramettres sont 
    """
    stra_stat=base_sondage[strate].value_counts()
    strates=stra_stat.index
    df=pd.DataFrame() # Dataframe devant contenir les échantillons par strate
    nbre_strate=len(strates)
    dic_strates={}
    poids_strate={}
    select=[] # Numéro des unités tirées 
    
    #calcul de la taille de l'échantillon par strate
    
    if share not in (0,1):
        print("L'option share doit être 0 ou 1")
        return select,df
    elif  share==0:
        for stra in strates:
            # échantillonnage pour la grappe 
            dic_strates[stra]=taille*stra_stat[stra]/stra_stat.sum()
        dic_strates=arrondi_hamilton(dic_strates)
        total_poids=pd.Series(dic_strates.values()).sum()
        if total_poids<taille:
            print(f"Les strates sont nombreuses et ne permettent pas de repartir assez bien l'echantillon de taille {taille}")
            return select,df
        else:
            for stra in strates:
                select_stra,df_ech_stra=select_sys(base_sondage[base_sondage[strate]==stra],
                                                   dic_strates[stra],
                                                   poids_u,
                                                   random_state
                                                  )
                df=pd.concat([df,df_ech_stra])
                select.extend(select_stra)
            return select,df
    elif share==1 and poids!=None:
        for stra in strates:
            dic_strates[stra]=stra_stat[stra]*base_sondage[base_sondage[strate]==stra][poids].sum()
        total_strates=pd.Series(dic_strates.values()).sum()
        for stra in strates:
            poids_strate[stra]=taille*dic_strates[stra]/total_strates
        poids_strate=arrondi_hamilton(poids_strate)
        total_poids=pd.Series(poids_strate.values()).sum()
        if total_poids<taille:
            print(f"Les strates sont nombreuses et ne permettent pas de repartir assez bien l'echantillon de taille {taille}")
            return select,df
        else:
            for stra in strates:
                select_stra,df_ech_stra=select_sys(base_sondage[base_sondage[strate]==stra],
                                               poids_strate[stra],
                                               poids_u,
                                               random_state
                                              )
                df=pd.concat([df,df_ech_stra])
                select.extend(select_stra)
            return select,df
    else:
        print("share=1 nécessite une variable de poids")
        return select,df

### **Test de l'implementation sur l'échantillonnage des ZD pour une étude dans la ville de Bafia au Cameroun**

In [1099]:
## importation de la base de sondage
base_sondage=pd.read_excel("./exemple/zd_bafia.xlsx")

In [1101]:
# visualisation des données
base_sondage.head()

,Region,Departemen,Arrondisse,Canton,Chefferie,Bloc_Loc,Ville,Milieu,ZD_4RGPH,ZD,ZE,ZS,ZC,Ménages_E,POP_E,Ménages_R,POP_R,N_A_R,Titre,ZD3
0,CENTRE,MBAM ET INOUBOU,BAFIA,BAFIA,BIABEREBE,BIABEREBE,NaN,1,33,4,AD,NaN,NaN,179,895,0,0,0,Arrondissement de BAFIA : ZD N° 5 ZE ( AD ),1
1,CENTRE,MBAM ET INOUBOU,BAFIA,BAFIA,BIABEREBE,BIABEREBE,NaN,1,40,16,AD,NaN,NaN,266,1170,0,0,0,Arrondissement de BAFIA : ZD N° 17 ZE ( AD ),1
2,CENTRE,MBAM ET INOUBOU,BAFIA,BAFIA,BIABEREBE,BIABEREBE,NaN,1,21,6,AB,NaN,NaN,237,1055,0,0,0,Arrondissement de BAFIA : ZD N° 7 ZE ( AB ),1
3,CENTRE,MBAM ET INOUBOU,BAFIA,BAFIA,BIABEREBE,BIABEREBE,NaN,1,51,7,AB,NaN,NaN,225,1277,0,0,0,Arrondissement de BAFIA : ZD N° 8 ZE ( AB ),1
4,CENTRE,MBAM ET INOUBOU,BAFIA,BAFIA,BIABEREBE,BIABEREBE,NaN,1,23,23,AL,NaN,NaN,167,965,0,0,0,Arrondissement de BAFIA : ZD N° 24 ZE ( AL ),1


In [1095]:
stra_stat=base_sondage["Chefferie"].value_counts()
strates=stra_stat.index
nbre_strate=len(strates)
dic_strates={}
poids_strate={}
select=[] # Numéro des unités tirées 
taille=60

for stra in strates:
    # échantillonnage pour la grappe 
    dic_strates[stra]=taille*stra_stat[stra]/stra_stat.sum() # round() ne permet pas d'avoir les arrondis classiques
dic_strates=arrondi_hamilton(dic_strates)
total_poids=sum(dic_strates.values())
print(total_poids)

60


### Probabilité égale

In [1103]:
select,df=select_sys(base_sondage[["Canton","Chefferie","ZD","Ménages_E","POP_E"]],random_state=0,taille=10)
df

,Canton,Chefferie,ZD,Ménages_E,POP_E,id_sys
3,BAFIA,BIABEREBE,7,225,1277,4
9,BAFIA,BIABETOM,14,182,852,10
15,BAFIA,BIABEYAKAN,5,270,1184,16
20,BAFIA,BIABEZOK,42,213,987,21
26,BAFIA,DOBATEM,44,281,953,27
32,BAFIA,LABLE,43,175,878,33
37,BAFIA,RIGAMA,15,247,1189,38
43,BAFIA,RIMIS I,25,248,1247,44
49,BAFIA,RIMIS II,3,203,836,50
54,BAFIA,RIMIS II,9,222,1126,55


### Probabilité inégale

In [1105]:
select,df=select_sys(base_sondage[["Canton","Chefferie","ZD","Ménages_E","POP_E"]],random_state=0,poids="POP_E",taille=10)
df

,Canton,Chefferie,ZD,Ménages_E,POP_E,id_sys,poids_relatif,p_inclusion,cum_p_inclusion
3,BAFIA,BIABEREBE,7,225,1277,4,0.021595,0.215954,0.743578
9,BAFIA,BIABETOM,14,182,852,10,0.014408,0.144082,1.699559
15,BAFIA,BIABEYAKAN,5,270,1184,16,0.020023,0.200227,2.708133
21,BAFIA,BIATSOTA,47,224,1346,22,0.022762,0.227622,3.796357
27,BAFIA,DOGUEM,55,167,999,28,0.016894,0.168941,4.804424
32,BAFIA,LABLE,43,175,878,33,0.014848,0.148479,5.715590
38,BAFIA,RIGAMA,56,173,851,39,0.014391,0.143913,6.774559
44,BAFIA,RIMIS I,29,258,1250,45,0.021139,0.211388,7.831498
49,BAFIA,RIMIS II,3,203,836,50,0.014138,0.141376,8.709519
54,BAFIA,RIMIS II,9,222,1126,55,0.019042,0.190418,9.653662


### Stratification proportionnelle (probabilité inégale)

In [1107]:
select,df=select_sys_complete(base_sondage[["Canton","Chefferie","Bloc_Loc","ZD","Ménages_E","POP_E"]],
                              random_state=0,
                              poids_u="POP_E",
                              poids="POP_E",
                              taille=10,
                              strate="Chefferie",
                              share=0
                             )
df

,Canton,Chefferie,Bloc_Loc,ZD,Ménages_E,POP_E,id_sys,poids_relatif,p_inclusion,cum_p_inclusion
42,BAFIA,RIMIS I,CHEFFERIE,26,196,913,4,0.097616,0.195231,0.802523
46,BAFIA,RIMIS I,CHEFFERIE,40,272,1190,8,0.127232,0.254464,1.801561
50,BAFIA,RIMIS II,CHEFFERIE,33,227,1116,3,0.126545,0.253090,0.726840
54,BAFIA,RIMIS II,CHEFFERIE,9,222,1126,7,0.127679,0.255358,1.739880
4,BAFIA,BIABEREBE,BIABEREBE,23,167,965,5,0.135059,0.135059,0.750455
11,BAFIA,BIABETOM,BIABETOM,31,203,991,5,0.150516,0.150516,0.737849
17,BAFIA,BIABEYAKAN,BIABEYAKAN,20,195,905,4,0.141694,0.141694,0.689526
37,BAFIA,RIGAMA,RIGAMA,15,247,1189,3,0.304637,0.304637,0.781963
31,KORRO,LABLE,ECOLE ANNEXE GROUPE 3,22,206,1114,2,0.355683,0.355683,0.719668
29,BAFIA,GONDON,GONDON,11,205,952,2,0.421986,0.421986,1.000000


### Stratification optimale (probabilité égale)

In [1109]:
select,df=select_sys_complete(base_sondage[["Canton","Chefferie","Bloc_Loc","ZD","Ménages_E","POP_E"]],
                              random_state=0,
                              poids="POP_E",
                              taille=10,
                              strate="Chefferie",
                              share=1
                             )
df

,Canton,Chefferie,Bloc_Loc,ZD,Ménages_E,POP_E,id_sys
40,BAFIA,RIMIS I,CHEFFERIE,41,244,1071,2
43,BAFIA,RIMIS I,CHEFFERIE,25,248,1247,5
46,BAFIA,RIMIS I,CHEFFERIE,40,272,1190,8
50,BAFIA,RIMIS II,CHEFFERIE,33,227,1116,3
54,BAFIA,RIMIS II,CHEFFERIE,9,222,1126,7
2,BAFIA,BIABEREBE,BIABEREBE,6,237,1055,3
5,BAFIA,BIABEREBE,BIABEREBE,8,224,979,6
11,BAFIA,BIABETOM,BIABETOM,31,203,991,5
17,BAFIA,BIABEYAKAN,BIABEYAKAN,20,195,905,4
37,BAFIA,RIGAMA,RIGAMA,15,247,1189,3


In [916]:
%pip install git+https://github.com/riozakies/echantillon.git

  Cloning https://github.com/riozakies/echantillon.git to /private/var/folders/l4/nkhcr2tn27q6rstw2p2vxqnm0000gn/T/pip-req-build-ol2uaeda
  Running command git clone --filter=blob:none --quiet https://github.com/riozakies/echantillon.git /private/var/folders/l4/nkhcr2tn27q6rstw2p2vxqnm0000gn/T/pip-req-build-ol2uaeda
  Resolved https://github.com/riozakies/echantillon.git to commit aa5ac5a5148a38eff76390b78c8945432006490b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for echantillon: filename=echantillon-0.1.0-py3-none-any.whl size=2916 sha256=ea3dbb82bfc6066cb3fdba30efac92c4f5bb9d4411d86856d55cb7b2c31ed4d5
  Stored in directory: /private/var/folders/l4/nkhcr2tn27q6rstw2p2vxqnm0000gn/T/pip-ephem-wheel-cache-3hcbxcao/wheels/2e/88/f0/831793e005916c30dc12b7c9c34923f0ee3e23c040e3565cb4
Successfully built echantillon
Note: you may need to restart the kernel to use updated packages.


In [920]:
import echantillon as ech

In [940]:
select,df=ech.select_sys_complete(base_sondage[["Canton","Chefferie","Bloc_Loc","ZD","Ménages_E","POP_E"]],
                              random_state=0,
                              poids="POP_E",
                              taille=28,
                              strate="Chefferie",
                              share=1
                             )
df

,Canton,Chefferie,Bloc_Loc,ZD,Ménages_E,POP_E,id_sys
39,BAFIA,RIMIS I,CHEFFERIE,38,210,892,1
41,BAFIA,RIMIS I,CHEFFERIE,27,165,877,3
42,BAFIA,RIMIS I,CHEFFERIE,26,196,913,4
43,BAFIA,RIMIS I,CHEFFERIE,25,248,1247,5
44,BAFIA,RIMIS I,CHEFFERIE,29,258,1250,6
46,BAFIA,RIMIS I,CHEFFERIE,40,272,1190,8
47,BAFIA,RIMIS I,CHEFFERIE,39,210,928,9
48,BAFIA,RIMIS II,CHEFFERIE,1,286,1253,1
50,BAFIA,RIMIS II,CHEFFERIE,33,227,1116,3
51,BAFIA,RIMIS II,CHEFFERIE,32,239,1275,4
